# 05 · Modelo M2: Sentence-BERT + LightGBM

Modelo intermedio entre M1 (TF-IDF, clásico) y M3 (DeBERTa con ajuste fino). La explicación se representa con **embeddings de Sentence-BERT congelados** (no se ajustan), que capturan el significado de la frase y no solo sus palabras, y un **LightGBM** aprende sobre ellos junto con variables tabulares de la pregunta y la respuesta.

**Ventajas esperadas:** se entrena en CPU en minutos, aprovecha un modelo de lenguaje preentrenado y combina de forma natural variables de texto con categóricas.

**Formulación:** plana, igual que M1 y M3: un solo clasificador sobre las 64 etiquetas `Category:Misconception`, cuyo top 3 sale de `predict_proba`.

| Bloque | Detalle |
|---|---|
| Embeddings | `all-MiniLM-L6-v2` (384 dim.) o `all-mpnet-base-v2` (768 dim.), normalizados. Se comparan dos textos: la entrada completa de M3 (`Question … Answer … Correct … Explanation …`) o solo la explicación |
| Variables tabulares | `QuestionId` y par pregunta-respuesta (categóricas), corrección de la respuesta, número de caracteres y de palabras, bandera `short_explanation` |
| Clasificador | LightGBM multiclase con *early stopping* en validación |

In [1]:
import json
import os
import sys
import time
import warnings
from pathlib import Path

RAIZ = Path.cwd()
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*eval_set.*")  # aviso de deprecación de LightGBM 4.7

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

from src import features_m2 as f2
from src import preprocesamiento as pp
from src.metricas import resumen_metricas, top_k
from src.utilidades import describir_hardware, guardar_predicciones, guardar_tuning, registrar_tiempo

NOMBRE = "M2_sbert_lgbm"
DIR_M2 = pp.DIR_MODELOS / "m2_sbert_lgbm"
DIR_CACHE = DIR_M2 / "cache"
DIR_CACHE.mkdir(parents=True, exist_ok=True)
SEMILLA = pp.SEMILLA
N_HILOS = max(1, (os.cpu_count() or 2) // 2)  # con todos los hilos lógicos LightGBM se vuelve ~30x más lento

print("lightgbm", lgb.__version__, "|", describir_hardware())

lightgbm 4.7.0 | CPU (x86_64)


## 1. Datos

Misma partición y misma función `preparar_dataframe` que usan M1, M3 y la aplicación.

In [2]:
df = pp.cargar_train_clean()
datos = pp.aplicar_particion(df)
respuestas, preguntas, misc_por_pregunta = pp.cargar_reglas()

prep = {s: pp.preparar_dataframe(datos[s], respuestas, preguntas) for s in ["train", "val", "test"]}
codigos = f2.codigos_categoricos(respuestas)
tabulares = {s: f2.variables_tabulares(prep[s], codigos) for s in prep}
y = {s: prep[s]["target"].values for s in prep}

{s: len(d) for s, d in prep.items()}

{'train': 25688, 'val': 3668, 'test': 7340}

In [3]:
pd.DataFrame(tabulares["train"][:5], columns=f2.COLUMNAS_TABULARES)

,pregunta_cod,pregunta_respuesta_cod,es_correcta_num,n_caracteres,n_palabras,short_explanation
0,0.0,8.0,1.0,61.0,16.0,0.0
1,0.0,8.0,1.0,31.0,7.0,0.0
2,0.0,8.0,1.0,62.0,15.0,0.0
3,0.0,8.0,1.0,53.0,12.0,0.0
4,0.0,8.0,1.0,52.0,10.0,0.0


## 2. Embeddings

Los embeddings se calculan una sola vez por combinación de encoder, texto y split, y se guardan en `models/m2_sbert_lgbm/cache/` para no recalcularlos durante el ajuste. Se registra el tiempo de cálculo, que forma parte del costo de entrenamiento e inferencia.

In [4]:
ENCODERS = {
    "minilm": "sentence-transformers/all-MiniLM-L6-v2",
    "mpnet": "sentence-transformers/all-mpnet-base-v2",
}
TEXTOS = ["entrada", "explicacion"]
tiempos_emb = {}  # (encoder, texto, split) -> segundos
_encoders = {}


def obtener_encoder(clave):
    if clave not in _encoders:
        _encoders[clave] = SentenceTransformer(ENCODERS[clave], device="cpu")
    return _encoders[clave]


def embeddings(clave, texto, split):
    ruta = DIR_CACHE / f"{clave}_{texto}_{split}.npy"
    ruta_t = ruta.with_suffix(".json")
    if ruta.exists():
        tiempos_emb[(clave, texto, split)] = json.loads(ruta_t.read_text())["segundos"]
        return np.load(ruta)
    inicio = time.time()
    emb = obtener_encoder(clave).encode(
        f2.textos_para_embedding(prep[split], texto), batch_size=128, normalize_embeddings=True
    ).astype(np.float32)
    seg = time.time() - inicio
    np.save(ruta, emb)
    ruta_t.write_text(json.dumps({"segundos": seg}))
    tiempos_emb[(clave, texto, split)] = seg
    return emb


for clave in ENCODERS:
    for texto in TEXTOS:
        for split in ["train", "val"]:
            emb = embeddings(clave, texto, split)
        print(f"{clave:7s} {texto:12s} dim={emb.shape[1]}  train+val: "
              f"{tiempos_emb[(clave, texto, 'train')] + tiempos_emb[(clave, texto, 'val')]:.0f} s")

minilm  entrada      dim=384  train+val: 230 s
minilm  explicacion  dim=384  train+val: 104 s


mpnet   entrada      dim=768  train+val: 1609 s


mpnet   explicacion  dim=768  train+val: 717 s


## 3. Ajuste de hiperparámetros (validación)

Igual que en M1 y M3, cada configuración se evalúa en validación con macro-F1 (métrica principal), exactitud y MAP@3, **sin** y **con** la máscara de etiquetas (que descarta las etiquetas incoherentes con la corrección de la respuesta y con las ideas erróneas posibles de la pregunta).

El *early stopping* usa la log-pérdida multiclase en validación (paciencia de 30 rondas), así que `n_estimators` es solo un tope. `max_bin=63` y `colsample_bytree=0.5` reducen el costo por ronda (cada ronda entrena 64 árboles, uno por clase) sin perder información relevante en embeddings continuos.

Para las clases con pocos ejemplos se prueba, además de sin pesos, una ponderación **√(n_max / n_c)** con tope de 10, la misma que funcionó mejor en M3. Los pesos `balanced` completos no se prueban porque en M1 empeoraron el macro-F1 en todas las combinaciones.

In [5]:
clases_train = np.array(sorted(np.unique(y["train"])))
mascara_val = pp.mascara_etiquetas(clases_train, prep["val"]["QuestionId"], prep["val"]["es_correcta"], misc_por_pregunta)

conteo = pd.Series(y["train"]).value_counts()
peso_clase_sqrt = np.minimum(np.sqrt(conteo.max() / conteo), 10.0)

BASE = dict(n_estimators=3000, learning_rate=0.1, num_leaves=31, min_child_samples=20,
            colsample_bytree=0.5, reg_lambda=1.0, max_bin=63)
resultados, modelos = [], {}


def matrices(clave, texto):
    return {s: f2.construir_matriz(embeddings(clave, texto, s), tabulares[s]) for s in ["train", "val"]}


def entrenar(X, params, pesos):
    clf = lgb.LGBMClassifier(objective="multiclass", random_state=SEMILLA, n_jobs=N_HILOS, verbose=-1,
                             **{**BASE, **params})
    w = None if pesos == "ninguno" else pd.Series(y["train"]).map(peso_clase_sqrt).values
    inicio = time.time()
    clf.fit(X["train"], y["train"], sample_weight=w,
            eval_set=[(X["val"], y["val"])], eval_metric="multi_logloss",
            categorical_feature=f2.CATEGORICAS,
            callbacks=[lgb.early_stopping(30, verbose=False)])
    return clf, time.time() - inicio


def registrar(clave, texto, params, pesos, X):
    clf, seg = entrenar(X, params, pesos)
    assert list(clf.classes_) == list(clases_train)
    proba = clf.predict_proba(X["val"])
    sin = resumen_metricas(y["val"], top_k(proba, clases_train)[0])
    con = resumen_metricas(y["val"], top_k(proba * mascara_val, clases_train)[0])
    fila = {"encoder": clave, "texto": texto, **{**BASE, **params}, "pesos": pesos,
            "mejor_iteracion": clf.best_iteration_, **con,
            "f1_macro_sin_mascara": sin["f1_macro"], "map_at_3_sin_mascara": sin["map_at_3"],
            "segundos_entrenamiento": round(seg, 1)}
    fila.pop("n_estimators")
    resultados.append(fila)
    modelos[len(resultados) - 1] = clf
    print(f"{clave:7s} {texto:12s} {params} pesos={pesos:8s} it={clf.best_iteration_:4d}  "
          f"F1={con['f1_macro']:.4f}  acc={con['accuracy']:.4f}  MAP@3={con['map_at_3']:.4f}  ({seg:.0f} s)")

### 3.1 Etapa 1: encoder y texto

Con los hiperparámetros base de LightGBM se comparan las 4 representaciones.

In [6]:
for clave in ENCODERS:
    for texto in TEXTOS:
        registrar(clave, texto, {}, "ninguno", matrices(clave, texto))

minilm  entrada      {} pesos=ninguno  it=  58  F1=0.5459  acc=0.8274  MAP@3=0.9074  (19 s)


minilm  explicacion  {} pesos=ninguno  it=  51  F1=0.4877  acc=0.8212  MAP@3=0.9043  (19 s)


mpnet   entrada      {} pesos=ninguno  it=  64  F1=0.5175  acc=0.8089  MAP@3=0.8972  (36 s)


mpnet   explicacion  {} pesos=ninguno  it=  55  F1=0.4954  acc=0.8203  MAP@3=0.9034  (39 s)


In [7]:
etapa1 = pd.DataFrame(resultados)
mejor_rep = etapa1.sort_values(["f1_macro", "map_at_3"], ascending=False).iloc[0]
ENC, TXT = mejor_rep["encoder"], mejor_rep["texto"]
print(f"Mejor representación: {ENC} + {TXT}")
etapa1[["encoder", "texto", "f1_macro", "accuracy", "map_at_3", "f1_macro_sin_mascara", "mejor_iteracion", "segundos_entrenamiento"]]

Mejor representación: minilm + entrada


,encoder,texto,f1_macro,accuracy,map_at_3,f1_macro_sin_mascara,mejor_iteracion,segundos_entrenamiento
0,minilm,entrada,0.545927,0.827426,0.907443,0.545422,58,19.4
1,minilm,explicacion,0.487685,0.821156,0.904262,0.487685,51,19.1
2,mpnet,entrada,0.517550,0.808888,0.897174,0.517550,64,36.0
3,mpnet,explicacion,0.495351,0.820338,0.903444,0.495351,55,39.3


### 3.2 Etapa 2: hiperparámetros de LightGBM

Sobre la mejor representación se ajustan primero la complejidad de los árboles y la tasa de aprendizaje, y después la regularización por hoja y los pesos de clase. Una búsqueda en dos pasos evita la explosión combinatoria (cada ajuste entrena 64 árboles por ronda).

In [8]:
X = matrices(ENC, TXT)
for num_leaves in [15, 31, 63]:
    for lr in [0.05, 0.1]:
        if (num_leaves, lr) == (31, 0.1):
            continue  # ya evaluada en la etapa 1
        registrar(ENC, TXT, {"num_leaves": num_leaves, "learning_rate": lr}, "ninguno", X)

minilm  entrada      {'num_leaves': 15, 'learning_rate': 0.05} pesos=ninguno  it= 122  F1=0.5728  acc=0.8282  MAP@3=0.9086  (28 s)


minilm  entrada      {'num_leaves': 15, 'learning_rate': 0.1} pesos=ninguno  it=  67  F1=0.5411  acc=0.8244  MAP@3=0.9059  (18 s)


minilm  entrada      {'num_leaves': 31, 'learning_rate': 0.05} pesos=ninguno  it= 107  F1=0.5694  acc=0.8340  MAP@3=0.9117  (28 s)


minilm  entrada      {'num_leaves': 63, 'learning_rate': 0.05} pesos=ninguno  it=  99  F1=0.5515  acc=0.8370  MAP@3=0.9133  (26 s)


minilm  entrada      {'num_leaves': 63, 'learning_rate': 0.1} pesos=ninguno  it=  51  F1=0.5415  acc=0.8274  MAP@3=0.9079  (19 s)


In [9]:
tabla = pd.DataFrame(resultados)
sub = tabla[(tabla.encoder == ENC) & (tabla.texto == TXT)]
paso1 = sub.sort_values(["f1_macro", "map_at_3"], ascending=False).iloc[0]
NL, LR = int(paso1["num_leaves"]), float(paso1["learning_rate"])
print(f"num_leaves={NL}, learning_rate={LR}")

for mcs in [10, 20, 40]:
    for pesos in ["ninguno", "sqrt"]:
        if mcs == 20 and pesos == "ninguno":
            continue  # ya evaluada en el paso anterior
        registrar(ENC, TXT, {"num_leaves": NL, "learning_rate": LR, "min_child_samples": mcs}, pesos, X)

num_leaves=15, learning_rate=0.05


minilm  entrada      {'num_leaves': 15, 'learning_rate': 0.05, 'min_child_samples': 10} pesos=ninguno  it= 119  F1=0.5581  acc=0.8277  MAP@3=0.9080  (27 s)


minilm  entrada      {'num_leaves': 15, 'learning_rate': 0.05, 'min_child_samples': 10} pesos=sqrt     it= 118  F1=0.5820  acc=0.8239  MAP@3=0.9065  (27 s)


minilm  entrada      {'num_leaves': 15, 'learning_rate': 0.05, 'min_child_samples': 20} pesos=sqrt     it= 117  F1=0.5862  acc=0.8277  MAP@3=0.9078  (27 s)


minilm  entrada      {'num_leaves': 15, 'learning_rate': 0.05, 'min_child_samples': 40} pesos=ninguno  it= 139  F1=0.5642  acc=0.8296  MAP@3=0.9094  (59 s)


minilm  entrada      {'num_leaves': 15, 'learning_rate': 0.05, 'min_child_samples': 40} pesos=sqrt     it= 120  F1=0.5892  acc=0.8282  MAP@3=0.9085  (28 s)


In [10]:
tabla = pd.DataFrame(resultados)
ruta_tuning = guardar_tuning(tabla, NOMBRE)
print("Guardado en", ruta_tuning.relative_to(RAIZ))
tabla.sort_values(["f1_macro", "map_at_3"], ascending=False)

Guardado en results/tuning/tuning_M2_sbert_lgbm.csv


,encoder,texto,learning_rate,num_leaves,min_child_samples,colsample_bytree,reg_lambda,max_bin,pesos,mejor_iteracion,f1_macro,accuracy,map_at_3,f1_macro_sin_mascara,map_at_3_sin_mascara,segundos_entrenamiento
13,minilm,entrada,0.05,15,40,0.5,1.0,63,sqrt,120,0.589210,0.828244,0.908533,0.589210,0.907988,27.5
11,minilm,entrada,0.05,15,20,0.5,1.0,63,sqrt,117,0.586169,0.827699,0.907761,0.586169,0.907306,27.3
10,minilm,entrada,0.05,15,10,0.5,1.0,63,sqrt,118,0.581986,0.823882,0.906534,0.581986,0.906080,26.8
4,minilm,entrada,0.05,15,20,0.5,1.0,63,ninguno,122,0.572756,0.828244,0.908624,0.572756,0.907443,27.7
6,minilm,entrada,0.05,31,20,0.5,1.0,63,ninguno,107,0.569426,0.833969,0.911714,0.569426,0.909578,27.8
12,minilm,entrada,0.05,15,40,0.5,1.0,63,ninguno,139,0.564243,0.829607,0.909351,0.564243,0.908715,59.4
9,minilm,entrada,0.05,15,10,0.5,1.0,63,ninguno,119,0.558138,0.827699,0.907988,0.558138,0.907125,27.4
7,minilm,entrada,0.05,63,20,0.5,1.0,63,ninguno,99,0.551462,0.836968,0.913350,0.551462,0.911169,26.3
0,minilm,entrada,0.10,31,20,0.5,1.0,63,ninguno,58,0.545927,0.827426,0.907443,0.545422,0.904807,19.4
8,minilm,entrada,0.10,63,20,0.5,1.0,63,ninguno,51,0.541539,0.827426,0.907852,0.541035,0.905125,19.0


## 4. Modelo final

Se elige la configuración con mayor macro-F1 en validación. Como el *early stopping* ya usó validación, se conserva el modelo de esa configuración tal como quedó (entrenado sobre `train`, con el número de árboles óptimo). Se guarda junto con una copia local del encoder, para que la aplicación no dependa de descargarlo.

In [11]:
idx = tabla.sort_values(["f1_macro", "map_at_3"], ascending=False).index[0]
mejor = tabla.loc[idx]
clf_final = modelos[idx]
print(mejor.to_string())

# La máscara se usa si no empeora el macro-F1 en validación
usar_mascara = bool(mejor["f1_macro"] >= mejor["f1_macro_sin_mascara"])
t_entrenamiento = mejor["segundos_entrenamiento"] + tiempos_emb[(ENC, TXT, "train")]
print(f"usar_mascara={usar_mascara} | entrenamiento (embeddings de train + LightGBM): {t_entrenamiento:.0f} s")

encoder                     minilm
texto                      entrada
learning_rate                 0.05
num_leaves                      15
min_child_samples               40
colsample_bytree               0.5
reg_lambda                     1.0
max_bin                         63
pesos                         sqrt
mejor_iteracion                120
f1_macro                   0.58921
accuracy                  0.828244
map_at_3                  0.908533
f1_macro_sin_mascara       0.58921
map_at_3_sin_mascara      0.907988
segundos_entrenamiento        27.5
usar_mascara=True | entrenamiento (embeddings de train + LightGBM): 229 s


In [12]:
joblib.dump(clf_final, DIR_M2 / "lgbm.joblib", compress=3)
obtener_encoder(ENC).save(str(DIR_M2 / "encoder"))
config = {
    "nombre": NOMBRE,
    "encoder": ENCODERS[ENC],
    "texto": TXT,
    "clases": clases_train.tolist(),
    "usar_mascara": usar_mascara,
    "codigos": codigos,
    "parametros": {k: (v.item() if hasattr(v, "item") else v) for k, v in clf_final.get_params().items()
                   if k in ["learning_rate", "num_leaves", "min_child_samples", "colsample_bytree", "reg_lambda", "max_bin"]},
    "pesos": mejor["pesos"],
    "mejor_iteracion": int(clf_final.best_iteration_),
    "metricas_val": {k: float(mejor[k]) for k in ["f1_macro", "accuracy", "map_at_3"]},
}
(DIR_M2 / "config_proyecto.json").write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
tam = sum(p.stat().st_size for p in DIR_M2.rglob("*") if p.is_file() and "cache" not in p.parts) / 1e6
print(f"Modelo guardado en {DIR_M2.relative_to(RAIZ)} ({tam:.0f} MB sin la caché)")

Modelo guardado en models/m2_sbert_lgbm (95 MB sin la caché)


## 5. Predicciones sobre el conjunto de prueba

Se generan con `src.prediccion.predecir_lote`, la misma función que usa la aplicación. El tiempo de inferencia incluye el cálculo de los embeddings, para compararlo de forma justa con M1 y M3.

In [13]:
from src import prediccion

inicio = time.time()
pred_test = prediccion.predecir_lote(NOMBRE, datos["test"])
t_inferencia = time.time() - inicio

ruta_pred = guardar_predicciones(pred_test, NOMBRE)
registrar_tiempo(NOMBRE, t_entrenamiento, t_inferencia)
print(f"{len(pred_test):,} predicciones en {t_inferencia:.1f} s -> {ruta_pred.relative_to(RAIZ)}")
pred_test.head()

7,340 predicciones en 56.1 s -> results/predicciones/pred_M2_sbert_lgbm.csv


,row_id,top1,top2,top3,prob1,prob2,prob3
0,0,True_Neither:NA,True_Correct:NA,True_Misconception:Incomplete,0.739681,0.258215,0.002020
1,16,True_Correct:NA,True_Neither:NA,True_Misconception:Incomplete,0.562134,0.437772,0.000047
2,19,True_Correct:NA,True_Neither:NA,True_Misconception:Incomplete,0.661092,0.338612,0.000241
3,24,True_Correct:NA,True_Neither:NA,True_Misconception:WNB,0.828780,0.171119,0.000084
4,26,True_Correct:NA,True_Neither:NA,True_Misconception:WNB,0.830788,0.169145,0.000046


In [14]:
# Verificación: predecir_lote reproduce en validación la métrica del ajuste
pred_val = prediccion.predecir_lote(NOMBRE, datos["val"])
resumen_metricas(datos["val"]["target"], pred_val[["top1", "top2", "top3"]].values)

{'f1_macro': 0.5892096580736775,
 'accuracy': 0.8282442748091603,
 'map_at_3': 0.9085332606324973}

## 6. Importancia de las variables

Ganancia total de LightGBM agrupada por bloque: cuánto aportan las variables tabulares frente a las dimensiones del embedding.

In [15]:
nombres = f2.nombres_columnas(clf_final.n_features_in_ - len(f2.COLUMNAS_TABULARES))
imp = pd.Series(clf_final.booster_.feature_importance("gain"), index=nombres)
bloques = pd.concat([imp[f2.COLUMNAS_TABULARES], pd.Series({"embedding (todas las dimensiones)": imp.drop(f2.COLUMNAS_TABULARES).sum()})])
(bloques / imp.sum() * 100).round(1).sort_values(ascending=False).rename("% de la ganancia")

embedding (todas las dimensiones)    54.6
pregunta_respuesta_cod               30.5
es_correcta_num                       8.0
pregunta_cod                          6.3
n_palabras                            0.4
n_caracteres                          0.2
short_explanation                     0.0
Name: % de la ganancia, dtype: float64

## 7. Ejemplo de uso desde la aplicación

In [16]:
prediccion.predecir(NOMBRE, 31772, r"\( \frac{3}{6} \)", "there are 3 white triangles and 6 blue ones so its 3/6")

[{'etiqueta': 'False_Misconception:WNB',
  'category': 'False_Misconception',
  'misconception': 'WNB',
  'prob': 0.989389},
 {'etiqueta': 'False_Neither:NA',
  'category': 'False_Neither',
  'misconception': 'NA',
  'prob': 0.009928},
 {'etiqueta': 'False_Misconception:Incomplete',
  'category': 'False_Misconception',
  'misconception': 'Incomplete',
  'prob': 0.000545}]